In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRanged,
    RandFlipd,
    RandRotate90d,
    RandAffined,
    SaveImaged
)

from monai.data import Dataset

data = [
    {
        "image": "CT",
        "label": "máscara"
    }
]

print("DATA PATH:", data)

transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    Orientationd(keys=["image", "label"], axcodes="RAS"),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 1.5),
        mode=("bilinear", "nearest")
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=-1000,
        a_max=1000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),

    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),

    RandAffined(
        keys=["image", "label"],
        prob=0.7,   # un poco más frecuente para variar más
        rotate_range=(0.2, 0.2, 0.2),
        scale_range=(0.15, 0.15, 0.15),
        mode=("bilinear", "nearest")
    ),
])

dataset = Dataset(data=data, transform=transforms)

# SAVE SETUP
for i in range(9):

    sample = dataset[0]

    saver = SaveImaged(
        keys=["image", "label"],
        output_dir="augmented_data",
        output_postfix=f"aug_{i}",
        separate_folder=False
    )

    saver(sample)

    print(f"Generated pair {i+1}")